In [3]:
import torch

# ตรวจสอบและใช้งาน GPU ของชิป M4
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"รันโมเดลบน: {device}")

รันโมเดลบน: mps


In [ ]:
import numpy as np
from scipy.io import loadmat
import gc
from torch.utils.data import DataLoader, TensorDataset
import torch

ai_model = 'transformer'
snr = 0
scenario = 'O1'
frequency = 28
antennas = 64

path = f'../DeepMIMO/DeepMIMO/DeepMIMO_dataset/SNR{snr}dB_{scenario}_{frequency}_Ant{antennas}/'
d1 = loadmat(path+'channel1.mat')['a']
d2 = loadmat(path+'channel2.mat')['b']
d3 = loadmat(path+'channel3.mat')['c']

# 2. รวมข้อมูล (ยังเป็น Complex อยู่)
data = np.concatenate((d1, d2, d3), axis=2).transpose(2, 0, 1)
del d1, d2, d3

# 3. แยก Real/Imag แล้วค่อยแปลงเป็น float32 (เพื่อไม่ให้ข้อมูลหาย!)
# ตรงนี้จะทำให้ได้ (Samples, 64, 64)
X_combined = np.concatenate((data.real, data.imag), axis=2).astype(np.float32)
X_flattened = np.reshape(X_combined, (data.shape[0], -1))

del data, X_combined
gc.collect()

# 4. Normalization แบบ Feature-wise (แม่นยำกว่า)
# ปรับสเกลแยกตามแต่ละ Antenna/Subcarrier
mean = np.mean(X_flattened, axis=0)
std = np.std(X_flattened, axis=0)
X_flattened = (X_flattened - mean) / (std + 1e-8)

# 5. โหลด Label
y = loadmat(path + 'DLCB_output.mat')['onehot_label'].astype(np.float32)

# 6. สร้าง Sequence
def create_sequences(data, labels, seq_length):
    num_samples = len(data) - seq_length
    X_seq = np.zeros((num_samples, seq_length, data.shape[1]), dtype=np.float32)
    y_seq = np.zeros((num_samples, labels.shape[1]), dtype=np.float32)
    for i in range(num_samples):
        X_seq[i] = data[i : i + seq_length]
        y_seq[i] = labels[i + seq_length - 1]
    return X_seq, y_seq

seq_length = 5
X_tn, y_tn = create_sequences(X_flattened, y, seq_length)
del X_flattened, y
gc.collect()

# 7. Sequential Split
split_idx = int(len(X_tn) * 0.7)
X_train, X_test = X_tn[:split_idx], X_tn[split_idx:]
y_train, y_test = y_tn[:split_idx], y_tn[split_idx:]

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=128, shuffle=False)

print(f"Fixed {ai_model.upper()} Input shape: {X_train.shape}")

KeyboardInterrupt: 

In [1]:
import torch
import torch.nn as nn
import math

# ==========================================
# [ Box 4: PositionalEncoding ]
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # ตรงกับกราฟ Box 4: รับ (1, 5, 22) -> ออก (1, 5, 22)
        x = x + self.pe[:x.size(1), :].unsqueeze(0)
        return x

# ==========================================
# [ Main Architecture: BeamPredictionTransformer ]
# ==========================================
class BeamPredictionTransformer(nn.Module):
    def __init__(self, input_size, d_model, num_heads, num_layers, n_beams, dropout_rate):
        super(BeamPredictionTransformer, self).__init__()
        
        # เตรียมกล่อง Linear (Box 2)
        self.embedding = nn.Linear(input_size, d_model)
        
        # เตรียมกล่อง PositionalEncoding (Box 4)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # เตรียมกล่อง TransformerEncoder (Box 5)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=num_heads, 
            dim_feedforward=d_model*4, 
            dropout=dropout_rate,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # เตรียมกล่อง Linear สำหรับ Output (Box 7)
        self.fc = nn.Linear(d_model, n_beams)

    def forward(self, x):
        # [ Box 1: input-tensor ]
        # เริ่มต้นรับข้อมูลเข้ามา Shape: (1, 5, 4096)
        
        # [ Box 2 & 3: Linear -> relu ]
        # บีบอัดมิติข้อมูลจาก 4096 เหลือ 22 (ตามค่า d_model)
        # Shape: (1, 5, 4096) -> เปลี่ยนเป็น -> (1, 5, 22)
        x = torch.relu(self.embedding(x)) 
        
        # [ Box 4: PositionalEncoding ]
        # ฝังตำแหน่งเวลาเข้าไปในข้อมูล (บวกค่าคงที่)
        # Shape: รับ (1, 5, 22) -> ออก (1, 5, 22) เท่าเดิม
        x = self.pos_encoder(x)
        
        # [ Box 5: TransformerEncoder ]
        # ส่งเข้ากลไก Multi-Head Attention ของ Transformer
        # Shape: รับ (1, 5, 22) -> ออก (1, 5, 22) เท่าเดิม
        x = self.transformer_encoder(x) 
        
        # [ Box 6: __getitem__ ]
        # การสไลซ์ (Slice) ดึงเฉพาะข้อมูล Time step สุดท้าย (Index -1)
        # นี่คือเหตุผลที่ torchview วาดกล่องชื่อ __getitem__ ขึ้นมา!
        # Shape: ถูกหั่นจาก (1, 5, 22) -> เหลือแค่ -> (1, 22)
        x = x[:, -1, :] 
        
        # [ Box 7 & 8: Linear -> output-tensor ]
        # โยนเข้าชั้น Classifier เพื่อให้โหวตเลือก Beam
        # Shape: ขยายจาก (1, 22) -> กลายเป็น -> (1, 64)
        out = self.fc(x) 
        
        return out

In [4]:
from sklearn.model_selection import train_test_split

model = BeamPredictionTransformer(input_size=4096, d_model=22, num_heads=2, num_layers=1, n_beams=64, dropout_rate=0.2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
criterion = nn.CrossEntropyLoss()

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Trainable Params: {params}')

Total Trainable Params: 97700


In [ ]:
import torch

model.eval()

dummy_input = torch.randn(1, 5, 4096).to(device)

onnx_file_path = "low_level_" + ai_model.lower()+".onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model successfully exported to {onnx_file_path}!")

In [ ]:
from torchview import draw_graph
import torch


dummy_input = torch.randn(1, 5, 4096)

model_graph = draw_graph(model, input_size=(1, 5, 4096), depth=1, expand_nested=False)
model_graph.visual_graph.render(f"high_level_{ai_model.lower()}", format="png")

In [ ]:
epochs = 100
best_loss = float('inf')
train_losses = []
val_losses = []
save_path = './best_models/'

for epoch in range(epochs):
    epoch_loss = 0.0
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        # Use torch.max to find the best index of beam for CrossEntropyLoss
        loss = criterion(outputs, torch.max(labels, 1)[1])
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    val_loss = 0.0
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, torch.max(labels, 1)[1])
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
        
    scheduler.step(avg_val_loss)
        
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth')
        print(f"--> Saved better model at Epoch {epoch+1} with Loss: {best_loss:.4f}")

        

In [ ]:
import evaluate as ev
import visualizer as vis

model.load_state_dict(torch.load(save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth'))

ds_config = {
    'snr': snr,
    'scenario': scenario,
    'frequency': frequency,
    'antennas': antennas
}  

mimo_results = ev.evaluate_performance(model, test_loader, device, criterion, ai_model)

vis.plot_training_loss(train_losses, val_losses, mimo_results, ds_config)

vis.plot_confusion_matrix(mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config)

vis.plot_beam_tracking(mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config)